In [1]:
import dotenv
import os, io, hashlib
from huggingface_hub import hf_hub_download
from pathlib import Path
from openai import OpenAI
import torchaudio
import torch
from pyannote.audio import Pipeline, Audio

In [2]:
# In der RENKU-Umgebung muss man nicht diese Zeile laufen lassen
dotenv.load_dotenv()

True

# Download Audio-File from the Huggingface-Repo

In [3]:
REPO_ID = "nkamiy/workshop2025-media"

SHA_WAV_RAMROD = os.getenv("SHA_WAV_RAMROD")

FILE = ["ramrod_ausschnitt_kurz.wav", f"sha256:{SHA_WAV_RAMROD}"]


def sha256sum(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


local = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILE[0],
        repo_type="dataset",   
        local_dir="../data"
    )

print(f"downloaded: {local}")

if FILE[1].startswith("sha256:"):
    got = sha256sum(Path(local))
    assert got == FILE[1].split(":", 1)[1], f"Checksum mismatch for {FILE[0]}"
print("All files ready.")

downloaded: ../data/ramrod_ausschnitt_kurz.wav
All files ready.


**Das folgende Zell braucht lange Laufzeit...**

In [4]:
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=os.getenv("HUGGINGFACE_ACCESS_TOKEN"))

audio_path = "../data/ramrod_ausschnitt_kurz.wav"
diarization = pipeline(audio_path, num_speakers=2)

for turn, _, speaker in diarization.itertracks(yield_label=True):
    print(f"start={turn.start:.1f}s stop={turn.end:.1f}s speaker_{speaker}")

/Users/nobu/openaiapi/.venv/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


start=5.1s stop=5.3s speaker_SPEAKER_00
start=7.5s stop=8.0s speaker_SPEAKER_00
start=9.3s stop=10.1s speaker_SPEAKER_01
start=10.9s stop=11.6s speaker_SPEAKER_01
start=13.3s stop=13.8s speaker_SPEAKER_00
start=15.6s stop=16.9s speaker_SPEAKER_01
start=29.6s stop=30.1s speaker_SPEAKER_00
start=32.4s stop=33.9s speaker_SPEAKER_00
start=35.4s stop=36.3s speaker_SPEAKER_01
start=37.4s stop=37.7s speaker_SPEAKER_00
start=38.8s stop=39.4s speaker_SPEAKER_01
start=44.1s stop=45.8s speaker_SPEAKER_00
start=47.6s stop=48.2s speaker_SPEAKER_00
start=51.5s stop=52.4s speaker_SPEAKER_01
start=54.9s stop=55.7s speaker_SPEAKER_00
start=57.1s stop=58.3s speaker_SPEAKER_00
start=59.1s stop=61.6s speaker_SPEAKER_00
start=62.6s stop=63.6s speaker_SPEAKER_01
start=64.6s stop=65.9s speaker_SPEAKER_01
start=67.9s stop=69.0s speaker_SPEAKER_00
start=73.9s stop=75.4s speaker_SPEAKER_01
start=78.0s stop=79.5s speaker_SPEAKER_00
start=81.3s stop=81.9s speaker_SPEAKER_01
start=83.2s stop=85.3s speaker_SPEAKER_

In [5]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

audio = Audio(sample_rate=16000, mono=True)

def audio_to_text(waveform: torch.Tensor, sample_rate: int) -> str:
    
    wav = waveform.detach().cpu().squeeze()  
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)               
    buf = io.BytesIO()
    torchaudio.save(buf, wav, sample_rate, format="wav")  
    buf.seek(0)

    file_tuple = ("segment.wav", buf.read(), "audio/wav")
    resp = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=file_tuple
    )
    return resp.text  # 


for segment, _, speaker in diarization.itertracks(yield_label=True):
    waveform, sample_rate = audio.crop(audio_path, segment)
    text = audio_to_text(waveform, sample_rate)
    print(f"[{segment.start:03.1f}s - {segment.end:03.1f}s] {speaker}: {text}")

[5.1s - 5.3s] SPEAKER_00: Hey.
[7.5s - 8.0s] SPEAKER_00: Hello Bill.
[9.3s - 10.1s] SPEAKER_01: Hello, Connie.
[10.9s - 11.6s] SPEAKER_01: Where is everybody?
[13.3s - 13.8s] SPEAKER_00: Rome
[15.6s - 16.9s] SPEAKER_01: Sure could do with a cup of coffee
[29.6s - 30.1s] SPEAKER_00: bill
[32.4s - 33.9s] SPEAKER_00: Did Burma really draw first?
[35.4s - 36.3s] SPEAKER_01: George said so, didn't he?
[37.4s - 37.7s] SPEAKER_00: City
[38.8s - 39.4s] SPEAKER_01: What do you think?
[44.1s - 45.8s] SPEAKER_00: I think you're not afraid to take a chance.
[47.6s - 48.2s] SPEAKER_00: I like this.
[51.5s - 52.4s] SPEAKER_01: What are you after?
[54.9s - 55.7s] SPEAKER_00: All right, OK.
[57.1s - 58.3s] SPEAKER_00: I want to break Frank Ivy.
[59.1s - 61.6s] SPEAKER_00: I want to see him crawl out of town the same way Walt did.
[62.6s - 63.6s] SPEAKER_01: If Curly dies
[64.6s - 65.9s] SPEAKER_01: Dave will take care of Ivy.
[67.9s - 69.0s] SPEAKER_00: Dave's way is too slow.
[73.9s - 75.4s] SPEAKER_

In [6]:
audio_file = open("../data/ramrod_ausschnitt_kurz.wav", "rb")
transcript = client.audio.transcriptions.create(
  model="gpt-4o-transcribe",
  file=audio_file
)

In [7]:
print(transcript.text)

Hello Bill. Hello Connie. Where is everybody? Around. Sure could do with a cup of coffee. Bill, did Burma really draw first? George said so, didn't he? Did he? Well, what do you think? I think you're not afraid to take a chance. I like that. What are you after, Connie? All right, I'll tell you. I want to break Frank Ivy. I want to see him crawl out of town the same way Walt did. If Curly dies, Dave will take care of Ivy. Dave's way is too slow. Just what's on your mind, Connie? I want you to stampede my herd. Stampede? It's the one way to make Dave and Jim Crue move fast. I thought about it for a long time. They'll think Frank Ivy did it. Yeah, they might. They will. This is something I didn't figure out. What did you figure, Bill? Well, to tell the truth, I... Why bother?
